# CHORUS Systems Atlas

This atlas explains how one locally generated night remains coherent while six rooms advance on one clock. It is an engineering companion to the prose documentation, not a player tutorial and not a substitute for the source.

The diagrams and tables below are computed from fixed architectural declarations. They are intended to make system boundaries inspectable without running the game.

In [1]:
from html import escape
from hashlib import sha256
import re

class HTMLResult(str):
    def _repr_html_(self):
        return str(self)

def table_html(caption, columns, rows, row_headers=False):
    head = "".join(f'<th scope="col">{escape(str(column))}</th>' for column in columns)
    body_rows = []
    for row in rows:
        rendered = []
        for index, value in enumerate(row):
            tag = "th" if row_headers and index == 0 else "td"
            scope = ' scope="row"' if tag == "th" else ""
            rendered.append(f'<{tag}{scope}>{escape(str(value))}</{tag}>')
        body_rows.append("<tr>" + "".join(rendered) + "</tr>")
    label = escape(caption)
    return HTMLResult(
        f'<div class="table-wrap" role="region" aria-label="{label}" tabindex="0">'
        f'<table><caption>{label}</caption><thead><tr>{head}</tr></thead>'
        f'<tbody>{"".join(body_rows)}</tbody></table></div>'
    )

def cards_html(title, cards):
    items = []
    for label, value, note in cards:
        items.append(
            '<article class="metric-card">'
            f'<h4>{escape(str(label))}</h4><strong>{escape(str(value))}</strong>'
            f'<p>{escape(str(note))}</p></article>'
        )
    return HTMLResult(f'<section class="metric-grid" aria-label="{escape(title)}">{"".join(items)}</section>')

def checklist_html(title, rows):
    items = []
    for status, label, evidence in rows:
        items.append(
            '<li>'
            f'<span class="status">{escape(status)}</span>'
            f'<strong>{escape(label)}</strong><p>{escape(evidence)}</p>'
            '</li>'
        )
    return HTMLResult(f'<section class="checklist" aria-label="{escape(title)}"><ul>{"".join(items)}</ul></section>')

DIAGRAM_STYLE = r"""
.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}
"""

SVG_COLORS = {
    "canvas": "#06130c",
    "group_fill": "#0b2819",
    "group_stroke": "#799886",
    "person_fill": "#163e2a",
    "person_stroke": "#a7e0bd",
    "system_fill": "#103522",
    "system_stroke": "#e2c57f",
    "component_fill": "#0d281a",
    "component_stroke": "#9bc8aa",
    "data_fill": "#262b22",
    "data_stroke": "#d5dedc",
    "evidence_fill": "#282433",
    "evidence_stroke": "#c9b6db",
    "boundary_fill": "#2d271b",
    "boundary_stroke": "#e2c57f",
    "risk_fill": "#351f1f",
    "risk_stroke": "#e0a8a8",
    "decision_fill": "#352b18",
    "decision_stroke": "#e2c57f",
    "title": "#f2f1e8",
    "body": "#c7d1c9",
    "role": "#a7e0bd",
    "flow": "#e2c57f",
    "data": "#a7e0bd",
    "evidence": "#c9b6db",
    "boundary": "#d5dedc",
    "association": "#b8c0bc",
    "label_bg": "#07140d",
    "label_stroke": "#5a6c60",
}


def dnode(node_id, x, y, w, h, title, body="", kind="component", shape="rect", role=""):
    return {
        "id": node_id, "x": float(x), "y": float(y), "w": float(w), "h": float(h),
        "title": title, "body": body, "kind": kind, "shape": shape, "role": role,
    }


def dedge(source, target, points, kind="flow", label="", label_at=None, arrow=True):
    return {
        "source": source, "target": target,
        "points": tuple((float(x), float(y)) for x, y in points),
        "kind": kind, "label": label, "label_at": label_at, "arrow": arrow,
    }


def dgroup(group_id, x, y, w, h, label, kind="boundary"):
    return {
        "id": group_id, "x": float(x), "y": float(y), "w": float(w),
        "h": float(h), "label": label, "kind": kind,
    }


def _slug(value):
    cleaned = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return cleaned or "diagram"


def _lines(value, max_chars, max_lines=4):
    raw_lines = str(value).split("\n") if value else []
    lines = []
    for raw in raw_lines:
        words = raw.split()
        if not words:
            lines.append("")
            continue
        current = words[0]
        for word in words[1:]:
            candidate = current + " " + word
            if len(candidate) <= max_chars:
                current = candidate
            else:
                lines.append(current)
                current = word
        lines.append(current)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1].rstrip(" …") + "…"
    return lines


def _segments(points):
    return list(zip(points, points[1:]))


def _on_boundary(point, node, tolerance=0.01):
    x, y = point
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    on_vertical = (
        (abs(x - left) <= tolerance or abs(x - right) <= tolerance)
        and top - tolerance <= y <= bottom + tolerance
    )
    on_horizontal = (
        (abs(y - top) <= tolerance or abs(y - bottom) <= tolerance)
        and left - tolerance <= x <= right + tolerance
    )
    return on_vertical or on_horizontal


def _segment_axis(segment):
    (x1, y1), (x2, y2) = segment
    if x1 == x2 and y1 != y2:
        return "v"
    if y1 == y2 and x1 != x2:
        return "h"
    raise AssertionError(f"Diagram route segment must be orthogonal and nonzero: {segment}")


def _segment_crosses_rect(segment, node):
    (x1, y1), (x2, y2) = segment
    left, top = node["x"], node["y"]
    right, bottom = left + node["w"], top + node["h"]
    axis = _segment_axis(segment)
    if axis == "h":
        if not top < y1 < bottom:
            return False
        return max(min(x1, x2), left) < min(max(x1, x2), right)
    if not left < x1 < right:
        return False
    return max(min(y1, y2), top) < min(max(y1, y2), bottom)


def _segment_intersection(first, second):
    a1, a2 = first
    b1, b2 = second
    axis_a, axis_b = _segment_axis(first), _segment_axis(second)
    if axis_a != axis_b:
        horizontal = first if axis_a == "h" else second
        vertical = second if axis_a == "h" else first
        (hx1, hy), (hx2, _) = horizontal
        (vx, vy1), (_, vy2) = vertical
        if min(hx1, hx2) <= vx <= max(hx1, hx2) and min(vy1, vy2) <= hy <= max(vy1, vy2):
            return (vx, hy)
        return None
    if axis_a == "h" and a1[1] == b1[1]:
        lo = max(min(a1[0], a2[0]), min(b1[0], b2[0]))
        hi = min(max(a1[0], a2[0]), max(b1[0], b2[0]))
        if lo < hi:
            return ("overlap", lo, hi, a1[1])
        if lo == hi:
            return (lo, a1[1])
    if axis_a == "v" and a1[0] == b1[0]:
        lo = max(min(a1[1], a2[1]), min(b1[1], b2[1]))
        hi = min(max(a1[1], a2[1]), max(b1[1], b2[1]))
        if lo < hi:
            return ("overlap", lo, hi, a1[0])
        if lo == hi:
            return (a1[0], lo)
    return None


def _rectangles_overlap(first, second):
    return (
        max(first["x"], second["x"]) < min(first["x"] + first["w"], second["x"] + second["w"])
        and max(first["y"], second["y"]) < min(first["y"] + first["h"], second["y"] + second["h"])
    )


def _validate_diagram(width, height, nodes, edges):
    assert width > 0 and height > 0
    node_map = {node["id"]: node for node in nodes}
    assert len(node_map) == len(nodes), "Diagram node IDs must be unique."
    for node in nodes:
        assert node["w"] > 0 and node["h"] > 0
        assert 0 <= node["x"] < width and 0 <= node["y"] < height
        assert node["x"] + node["w"] <= width and node["y"] + node["h"] <= height
    for index, first in enumerate(nodes):
        for second in nodes[index + 1:]:
            assert not _rectangles_overlap(first, second), (
                f"Diagram nodes overlap: {first['id']} and {second['id']}"
            )

    all_segments = []
    for edge_index, edge in enumerate(edges):
        assert edge["source"] in node_map and edge["target"] in node_map
        points = edge["points"]
        assert len(points) >= 2
        assert _on_boundary(points[0], node_map[edge["source"]]), (
            f"Route must start on source boundary: {edge}"
        )
        assert _on_boundary(points[-1], node_map[edge["target"]]), (
            f"Route must end on target boundary: {edge}"
        )
        for segment_index, segment in enumerate(_segments(points)):
            _segment_axis(segment)
            for node_id, node in node_map.items():
                if node_id in (edge["source"], edge["target"]):
                    continue
                assert not _segment_crosses_rect(segment, node), (
                    f"Route crosses node {node_id}: {edge}"
                )
            all_segments.append((edge_index, segment_index, edge, segment))

    for index, first in enumerate(all_segments):
        for second in all_segments[index + 1:]:
            edge_a, edge_b = first[2], second[2]
            if first[0] == second[0]:
                continue
            intersection = _segment_intersection(first[3], second[3])
            if intersection is None:
                continue
            shared_terminal_points = (
                set((edge_a["points"][0], edge_a["points"][-1]))
                & set((edge_b["points"][0], edge_b["points"][-1]))
            )
            if (
                isinstance(intersection, tuple)
                and intersection
                and intersection[0] != "overlap"
                and intersection in shared_terminal_points
            ):
                continue
            raise AssertionError(
                f"Diagram routes cross or overlap at {intersection}: {edge_a} / {edge_b}"
            )
    return {
        "nodes": len(nodes), "edges": len(edges), "segments": len(all_segments),
        "crossings": 0, "node_incursions": 0, "node_overlaps": 0,
    }


def _svg_text(x, y, lines, fill, size, weight=400, line_height=15, anchor="start", letter_spacing=0):
    if not lines:
        return ""
    spans = []
    for index, line in enumerate(lines):
        dy = 0 if index == 0 else line_height
        spans.append(f'<tspan x="{x:g}" dy="{dy:g}">{escape(line)}</tspan>')
    return (
        f'<text x="{x:g}" y="{y:g}" fill="{fill}" font-size="{size:g}" '
        f'font-weight="{weight}" text-anchor="{anchor}" letter-spacing="{letter_spacing:g}" '
        'font-family="Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif">'
        f'{"".join(spans)}</text>'
    )


def _node_colors(kind):
    return (
        SVG_COLORS.get(f"{kind}_fill", SVG_COLORS["component_fill"]),
        SVG_COLORS.get(f"{kind}_stroke", SVG_COLORS["component_stroke"]),
    )


def _edge_dash(kind):
    if kind == "evidence":
        return ' stroke-dasharray="7 5"'
    if kind == "boundary":
        return ' stroke-dasharray="3 5"'
    return ""


def _figure_shell(diagram_type, title, description, svg, assurance, legend=(), notes=(), equivalent=""):
    legend_html = ""
    if legend:
        legend_items = ''.join(
            f'<li><i class="diagram-key diagram-key-{escape(kind)}" aria-hidden="true"></i>'
            f'<span>{escape(label)}</span></li>'
            for kind, label in legend
        )
        legend_html = f'<ul class="diagram-legend" aria-label="Diagram legend">{legend_items}</ul>'
    notes_html = ''.join(f'<li>{escape(str(note))}</li>' for note in notes)
    if notes_html:
        notes_html = f'<ul class="diagram-notes">{notes_html}</ul>'
    return HTMLResult(
        f'<figure class="diagram-figure" data-diagram-type="{escape(diagram_type)}" '
        'data-routing="orthogonal-crossing-free">'
        f'<figcaption><span>{escape(diagram_type)}</span><strong>{escape(title)}</strong>'
        f'<p>{escape(description)}</p></figcaption>'
        f'<div class="diagram-canvas" role="region" aria-label="{escape(title)} diagram" tabindex="0">{svg}</div>'
        f'{legend_html}{notes_html}<span class="diagram-assurance">{escape(assurance)}</span>{equivalent}'
        '</figure>'
    )


def diagram_html(diagram_type, title, description, width, height, nodes, edges, groups=(), legend=(), notes=()):
    nodes = tuple(nodes)
    edges = tuple(edges)
    groups = tuple(groups)
    assurance = _validate_diagram(width, height, nodes, edges)
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    node_map = {node["id"]: node for node in nodes}

    marker_kinds = sorted(set(edge.get("kind", "flow") for edge in edges if edge.get("arrow", True)))
    defs = []
    for kind in marker_kinds:
        marker = uid + "-arrow-" + _slug(kind)
        color = SVG_COLORS.get(kind, SVG_COLORS["flow"])
        defs.append(
            f'<marker id="{marker}" viewBox="0 0 10 10" refX="9" refY="5" '
            'markerWidth="7" markerHeight="7" orient="auto-start-reverse">'
            f'<path d="M 0 0 L 10 5 L 0 10 z" fill="{color}"/></marker>'
        )

    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" '
        f'viewBox="0 0 {width:g} {height:g}" role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
        f'<defs>{"".join(defs)}</defs>',
    ]

    for group in groups:
        parts.append(
            f'<rect x="{group["x"]:g}" y="{group["y"]:g}" width="{group["w"]:g}" '
            f'height="{group["h"]:g}" rx="18" fill="{SVG_COLORS["group_fill"]}" fill-opacity=".28" '
            f'stroke="{SVG_COLORS["group_stroke"]}" stroke-width="1.2" stroke-dasharray="7 5"/>'
        )
        parts.append(
            _svg_text(group["x"] + 14, group["y"] + 21, [group["label"]], SVG_COLORS["title"], 12, 700, 14, "start", .7)
        )

    for edge in edges:
        points = " ".join(f'{x:g},{y:g}' for x, y in edge["points"])
        color = SVG_COLORS.get(edge["kind"], SVG_COLORS["flow"])
        marker_attr = ""
        if edge.get("arrow", True):
            marker_attr = f' marker-end="url(#{uid}-arrow-{_slug(edge["kind"])})"'
        parts.append(
            f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2" '
            f'stroke-linecap="round" stroke-linejoin="round"{_edge_dash(edge["kind"])}{marker_attr}/>'
        )
        if edge.get("label"):
            lx, ly = edge.get("label_at") or edge["points"][len(edge["points"]) // 2]
            label_width = max(72, min(150, len(edge["label"]) * 6.4 + 18))
            parts.append(
                f'<rect x="{lx - label_width / 2:g}" y="{ly - 12:g}" width="{label_width:g}" height="22" '
                f'rx="7" fill="{SVG_COLORS["label_bg"]}" stroke="{SVG_COLORS["label_stroke"]}" stroke-width=".8"/>'
            )
            parts.append(_svg_text(lx, ly + 3, [edge["label"]], SVG_COLORS["title"], 10, 700, 12, "middle"))

    for node in nodes:
        x, y, w, h = node["x"], node["y"], node["w"], node["h"]
        shape = node.get("shape", "rect")
        fill, stroke = _node_colors(node.get("kind", "component"))
        common = f'fill="{fill}" stroke="{stroke}" stroke-width="1.6"'
        if shape == "diamond":
            points = f'{x + w / 2:g},{y:g} {x + w:g},{y + h / 2:g} {x + w / 2:g},{y + h:g} {x:g},{y + h / 2:g}'
            parts.append(f'<polygon points="{points}" {common}/>' )
        elif shape == "pill":
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="{h / 2:g}" {common}/>' )
        elif shape == "document":
            fold = min(18, w * .12)
            d = (
                f'M {x:g} {y:g} H {x + w - fold:g} L {x + w:g} {y + fold:g} '
                f'V {y + h:g} H {x:g} Z M {x + w - fold:g} {y:g} V {y + fold:g} H {x + w:g}'
            )
            parts.append(f'<path d="{d}" {common} stroke-linejoin="round"/>' )
        else:
            parts.append(f'<rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="12" {common}/>' )

        char_width = max(13, int((w - 24) / 7.1))
        title_lines = _lines(node["title"], char_width, 2)
        body_lines = _lines(node.get("body", ""), char_width, 4)
        role = node.get("role", "")
        top = y + 20
        if role:
            parts.append(_svg_text(x + w / 2, top, [role.upper()], SVG_COLORS["role"], 10, 700, 12, "middle", .8))
            top += 18
        parts.append(_svg_text(x + w / 2, top, title_lines, SVG_COLORS["title"], 14, 700, 16, "middle"))
        body_y = top + 16 * len(title_lines) + 5
        parts.append(_svg_text(x + w / 2, body_y, body_lines, SVG_COLORS["body"], 11.5, 400, 14, "middle"))

    parts.append('</svg>')
    relation_items = []
    for edge in edges:
        relation = f'{node_map[edge["source"]]["title"]} → {node_map[edge["target"]]["title"]}'
        if edge.get("label"):
            relation += f' ({edge["label"]})'
        relation_items.append(f'<li>{escape(relation)}</li>')
    node_items = [
        f'<li><strong>{escape(node["title"])}</strong>'
        f'{": " + escape(node["body"]) if node.get("body") else ""}</li>'
        for node in nodes
    ]
    equivalent = (
        '<details class="diagram-equivalent"><summary>Text equivalent</summary>'
        f'<h4>Elements</h4><ul>{"".join(node_items)}</ul>'
        f'<h4>Relationships</h4><ul>{"".join(relation_items) if relation_items else "<li>No connector relationships; the diagram uses nested evidentiary zones.</li>"}</ul>'
        '</details>'
    )
    assurance_text = (
        f'Validated: {assurance["nodes"]} nodes · {assurance["edges"]} edges · '
        'orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps'
    )
    return _figure_shell(
        diagram_type, title, description, ''.join(parts), assurance_text,
        legend=legend, notes=notes, equivalent=equivalent,
    )


def matrix_diagram_html(diagram_type, title, description, rows, columns, coverage, notes=()):
    rows = tuple(rows)
    columns = tuple(columns)
    valid_marks = {"P", "S", ""}
    assert len(set(rows)) == len(rows) and len(set(columns)) == len(columns)
    for key, mark in coverage.items():
        assert key[0] in rows and key[1] in columns and mark in valid_marks

    left = 255
    top = 125
    cell_w = 125
    cell_h = 72
    right_pad = 25
    bottom_pad = 35
    width = left + cell_w * len(columns) + right_pad
    height = top + cell_h * len(rows) + bottom_pad
    uid = _slug(title) + "-" + sha256(title.encode("utf-8")).hexdigest()[:8]
    title_id, desc_id = uid + "-title", uid + "-desc"
    parts = [
        f'<svg class="diagram-svg" width="{width:g}" height="{height:g}" viewBox="0 0 {width:g} {height:g}" '
        f'role="img" aria-labelledby="{title_id} {desc_id}">',
        f'<title id="{title_id}">{escape(title)}</title>',
        f'<desc id="{desc_id}">{escape(description)}</desc>',
        f'<rect x="0" y="0" width="{width:g}" height="{height:g}" fill="{SVG_COLORS["canvas"]}"/>',
    ]
    for column_index, column in enumerate(columns):
        x = left + column_index * cell_w
        lines = _lines(column, 15, 3)
        parts.append(_svg_text(x + cell_w / 2, 38, lines, SVG_COLORS["title"], 11, 700, 14, "middle"))
    for row_index, row in enumerate(rows):
        y = top + row_index * cell_h
        parts.append(
            f'<rect x="8" y="{y:g}" width="{left - 16:g}" height="{cell_h:g}" rx="8" '
            f'fill="{SVG_COLORS["component_fill"]}" stroke="{SVG_COLORS["component_stroke"]}" stroke-width="1"/>'
        )
        parts.append(_svg_text(20, y + 28, _lines(row, 30, 2), SVG_COLORS["title"], 12, 700, 15, "start"))
        for column_index, column in enumerate(columns):
            x = left + column_index * cell_w
            mark = coverage.get((row, column), "")
            if mark == "P":
                fill, stroke, label = "#173d29", "#a7e0bd", "P"
            elif mark == "S":
                fill, stroke, label = "#2b2835", "#c9b6db", "S"
            else:
                fill, stroke, label = "#0a1a11", "#38483e", "—"
            parts.append(
                f'<rect x="{x:g}" y="{y:g}" width="{cell_w:g}" height="{cell_h:g}" '
                f'fill="{fill}" stroke="{stroke}" stroke-width="1"/>'
            )
            parts.append(_svg_text(x + cell_w / 2, y + 42, [label], SVG_COLORS["title"] if mark else SVG_COLORS["body"], 17, 700, 18, "middle"))
    parts.append('</svg>')

    table_rows = []
    for row in rows:
        table_rows.append((row, *({"P": "Primary", "S": "Supporting", "": "Not claimed"}[coverage.get((row, column), "")] for column in columns)))
    equivalent = str(table_html(
        title + " text equivalent",
        ("Test family", *columns),
        table_rows,
        row_headers=True,
    ))
    equivalent = f'<details class="diagram-equivalent"><summary>Text equivalent</summary>{equivalent}</details>'
    return _figure_shell(
        diagram_type, title, description, ''.join(parts),
        f'Validated: {len(rows)} test families · {len(columns)} concern columns · matrix topology · 0 connector lines',
        legend=(("data", "P = primary coverage"), ("evidence", "S = supporting coverage")),
        notes=notes,
        equivalent=equivalent,
    )


_html = HTMLResult(f"<style>{DIAGRAM_STYLE}</style>")
print("Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.")
_html

Standard-library rendering helpers loaded, including deterministic SVG diagrams with crossing validation.


.diagram-figure{margin:1rem 0;padding:1rem;border:1px solid rgba(213,222,220,.24);border-radius:.8rem 1.3rem .9rem 1.1rem;background:rgba(4,16,10,.76)}
.diagram-figure figcaption{display:grid;gap:.3rem;margin-bottom:.8rem}.diagram-figure figcaption span{color:#e2c57f;font:700 .72rem/1.35 ui-monospace,SFMono-Regular,Consolas,monospace;letter-spacing:.08em;text-transform:uppercase}.diagram-figure figcaption strong{font-size:1.25rem;color:#f2f1e8}.diagram-figure figcaption p{max-width:78ch;margin:0;color:#c7d1c9}
.diagram-canvas{max-width:100%;overflow-x:auto;border:1px solid rgba(213,222,220,.16);border-radius:.65rem;background:#06130c}.diagram-svg{display:block;width:100%;min-width:760px;height:auto}.diagram-legend{display:flex;flex-wrap:wrap;gap:.5rem 1rem;margin:.8rem 0 0;padding:0;list-style:none;color:#c7d1c9;font-size:.8rem}.diagram-legend li{display:flex;align-items:center;gap:.4rem}.diagram-key{width:1.8rem;height:.2rem;display:inline-block;background:#e2c57f}.diagram-key-data{background:#a7e0bd}.diagram-key-evidence{height:0;border-top:2px dashed #c9b6db;background:none}.diagram-key-boundary{height:0;border-top:2px dashed #d5dedc;background:none}.diagram-key-association{background:#b8c0bc}.diagram-assurance{display:block;margin-top:.7rem;color:#a7e0bd;font:700 .72rem/1.4 ui-monospace,SFMono-Regular,Consolas,monospace}.diagram-equivalent{margin-top:.7rem;border-top:1px solid rgba(213,222,220,.18)}.diagram-equivalent summary{min-height:44px;padding:.7rem 0;cursor:pointer;color:#d5dedc}.diagram-equivalent h4{margin:.7rem 0 .3rem;color:#e2c57f}.diagram-equivalent ul{margin:.2rem 0 0;padding-left:1.2rem}.diagram-notes{color:#c7d1c9}
@media(max-width:42rem){.diagram-figure{padding:.65rem}.diagram-svg{min-width:700px}}
@media(forced-colors:active){.diagram-figure,.diagram-canvas{border:1px solid CanvasText}}

## Source provenance

The atlas is tied to the implementation rather than an independently maintained diagram. This executed cell reads the authoritative modules, records their line counts and short content digests, and extracts the generator version that governs deterministic nights.

In [2]:
from hashlib import sha256
from pathlib import Path
import re

def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "app" / "scenario-generator.ts").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from within the CHORUS source tree.")

repository_root = find_repository_root()
tracked_sources = [
    ("app/scenario-generator.ts", "Validated night grammar and coherence gates"),
    ("app/night-engine.ts", "Shared-clock event reducer and receipt validation"),
    ("app/page.tsx", "One-viewport disclosure and interaction shell"),
    ("app/debrief-copy.ts", "Pure played-path summary and concept derivation"),
    ("app/save-model.ts", "Portable-state and local-slot validation"),
    ("app/privacy-panel.tsx", "Player-directed persistence controls"),
]
provenance_rows = []
generator_text = ""
for relative_path, role in tracked_sources:
    payload = (repository_root / relative_path).read_bytes()
    source_text = payload.decode("utf-8")
    if relative_path == "app/scenario-generator.ts":
        generator_text = source_text
    provenance_rows.append((relative_path, len(source_text.splitlines()), sha256(payload).hexdigest()[:12], role))

version_match = re.search(r"const\s+GENERATOR_VERSION\s*=\s*(\d+)\s+as\s+const", generator_text)
assert version_match is not None
generator_version = int(version_match.group(1))
assert generator_version > 0 and len(provenance_rows) == len(tracked_sources)
_html = table_html("Implementation provenance", ("Source", "Lines", "SHA-256 (12)", "Authority"), provenance_rows, row_headers=True)
print(f"PASS: read {len(provenance_rows)} authoritative modules; parsed generator version {generator_version}.")
_html

PASS: read 6 authoritative modules; parsed generator version 14.


Source,Lines,SHA-256 (12),Authority
app/scenario-generator.ts,5065,3f594f938f35,Validated night grammar and coherence gates
app/night-engine.ts,1283,fcbc5abf6a0b,Shared-clock event reducer and receipt validation
app/page.tsx,923,51f3a2278a26,One-viewport disclosure and interaction shell
app/debrief-copy.ts,1161,db7ea3d6bbb9,Pure played-path summary and concept derivation
app/save-model.ts,1105,bd9f017fdace,Portable-state and local-slot validation
app/privacy-panel.tsx,260,6d54b600b68f,Player-directed persistence controls


## Runtime topology

The runtime is intentionally narrow: generation establishes a validated world, the event reducer is the sole authority for change, and the interface discloses only what the occupied seat may currently know. Persistence stores the same event-sourced state instead of creating a parallel model.

In [3]:
modules = [
    ("Scenario grammar", "app/scenario-generator.ts", "Creates six reviewed rooms, links, scenes, choices, ledgers, and coherence reports."),
    ("Night reducer", "app/night-engine.ts", "Applies decisions, time advances, ambient pulses, access checks, and cross-room receipts."),
	    ("Interface", "app/page.tsx", "Renders the bounded house, room switching, progressive disclosure, relations, and the completed-night reading order."),
	    ("Conclusion copy", "app/debrief-copy.ts", "Derives the route-faithful natural summary and evidence-qualified plain concept receipt from typed pack and state records."),
	    ("Persistence", "app/save-model.ts", "Validates local slots and portable state with provenance, size, schema, and digest checks."),
	    ("Privacy controls", "app/privacy-panel.tsx", "Keeps open-tab memory as the default and exposes explicit save, import, export, and deletion controls."),
]
_html = table_html("Primary runtime modules", ("Responsibility", "Source", "Boundary"), modules, row_headers=True)
print(f"Mapped {len(modules)} primary modules; state changes remain concentrated in the generator and reducer.")
_html

Mapped 6 primary modules; state changes remain concentrated in the generator and reducer.


Responsibility,Source,Boundary
Scenario grammar,app/scenario-generator.ts,"Creates six reviewed rooms, links, scenes, choices, ledgers, and coherence reports."
Night reducer,app/night-engine.ts,"Applies decisions, time advances, ambient pulses, access checks, and cross-room receipts."
Interface,app/page.tsx,"Renders the bounded house, room switching, progressive disclosure, relations, and the completed-night reading order."
Conclusion copy,app/debrief-copy.ts,Derives the route-faithful natural summary and evidence-qualified plain concept receipt from typed pack and state records.
Persistence,app/save-model.ts,"Validates local slots and portable state with provenance, size, schema, and digest checks."
Privacy controls,app/privacy-panel.tsx,"Keeps open-tab memory as the default and exposes explicit save, import, export, and deletion controls."


In [4]:
flow = [
    (1, "Seed", "One unsigned 32-bit value", "No state mutation"),
    (2, "Validated night pack", "Rooms, truth, language, routes, beats, choices", "Generation boundary"),
    (3, "Night state", "Clock, room runtimes, decisions, pulses", "Reducer-owned"),
    (4, "Visible house", "Current seat plus bounded cross-room cues", "Disclosure policy"),
    (5, "Portable state", "Versioned envelope and deterministic digest", "Player-controlled"),
]
_html = table_html("State flow from seed to portable record", ("Order", "Stage", "Contents", "Control"), flow)
print("State flow verified: presentation does not become a second source of truth.")
_html

State flow verified: presentation does not become a second source of truth.


Order,Stage,Contents,Control
1,Seed,One unsigned 32-bit value,No state mutation
2,Validated night pack,"Rooms, truth, language, routes, beats, choices",Generation boundary
3,Night state,"Clock, room runtimes, decisions, pulses",Reducer-owned
4,Visible house,Current seat plus bounded cross-room cues,Disclosure policy
5,Portable state,Versioned envelope and deterministic digest,Player-controlled


### Layered technical architecture

The layered view separates generation, runtime state, interaction surfaces, and assurance/publication. Vertical alignment names the principal dependency chain while preserving the reducer as the sole runtime authority.

In [5]:
nodes=[
 dnode('sg',70,100,240,90,'Scenario grammar','Incidents, routes, artifacts, choices','component','rect','generation'),
 dnode('ag',380,100,240,90,'Actor generation','Motives, affect, ties, repertoires','component','rect','generation'),
 dnode('cg',690,100,240,90,'Coherence gates','Schema, agency, ethics, replay readiness','evidence','rect','generation'),
 dnode('ne',70,290,240,90,'Night engine','Clock, access, reducer, completion','component','rect','runtime'),
 dnode('ce',380,290,240,90,'Cross-room effects','Directed ambient and direct receipts','component','rect','runtime'),
 dnode('sp',690,290,240,90,'State and persistence','Night state, save schema, digest','data','rect','runtime'),
 dnode('ui',70,480,240,90,'UI shell','One viewport and room navigation','system','rect','surface'),
 dnode('tr',380,480,240,90,'Trace / Relations / Receipts','Bounded inspection and interpretation','system','rect','surface'),
 dnode('hg',690,480,240,90,'House Guide / Privacy','Help, scholarship, local controls','system','rect','surface'),
 dnode('ft',70,670,240,90,'Focused tests','Generation, runtime, save, a11y, viewport','evidence','rect','assurance'),
 dnode('ev',380,670,240,90,'Evidence records','Simulation runs, status, route checks','evidence','document','assurance'),
 dnode('nb',690,670,240,90,'Notebook publication','Executed sources, HTML, manifests','evidence','document','publication'),
]
edges=[
 dedge('sg','ne',[(190,190),(190,290)],'data'),dedge('ag','ce',[(500,190),(500,290)],'data'),dedge('cg','sp',[(810,190),(810,290)],'evidence'),
 dedge('ne','ui',[(190,380),(190,480)],'flow'),dedge('ce','tr',[(500,380),(500,480)],'data'),dedge('sp','hg',[(810,380),(810,480)],'data'),
 dedge('ui','ft',[(190,570),(190,670)],'evidence'),dedge('tr','ev',[(500,570),(500,670)],'evidence'),dedge('hg','nb',[(810,570),(810,670)],'evidence'),
]
_html = diagram_html('Layered technical architecture','Layered CHORUS technical architecture','The architecture is organized as generation, runtime, player-facing surfaces, and assurance/publication. Vertical alignment shows the principal dependency chain without implying that presentation owns model state.',1000,810,nodes,edges,groups=[dgroup('lg',25,55,950,155,'Generation layer'),dgroup('lr',25,245,950,155,'Runtime and state layer'),dgroup('ls',25,435,950,155,'Interaction and disclosure layer'),dgroup('la',25,625,950,155,'Assurance and publication layer')],legend=[('data','Data/state dependency'),('flow','Runtime control'),('evidence','Verification/publication dependency')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Layered technical architecture  Layered CHORUS technical architecture  The architecture is organized as generation, runtime, player-facing surfaces, and assurance/publication. Vertical alignment shows the principal dependency chain without implying that presentation owns model state.     Layered CHORUS technical architecture  The architecture is organized as generation, runtime, player-facing surfaces, and assurance/publication. Vertical alignment shows the principal dependency chain without implying that presentation owns model state.                Generation layer     Runtime and state layer     Interaction and disclosure layer     Assurance and publication layer              GENERATION    Scenario grammar    Incidents, routes, artifacts,  choices     GENERATION    Actor generation    Motives, affect, ties,  repertoires     GENERATION    Coherence gates    Schema, agency, ethics, replay  readiness     RUNTIME    Night engine    Clock, access, reducer,  completion     RUNTIME    Cross-room effects    Directed ambient and direct  receipts     RUNTIME    State and persistence    Night state, save schema,  digest     SURFACE    UI shell    One viewport and room  navigation     SURFACE    Trace / Relations / Receipts    Bounded inspection and  interpretation     SURFACE    House Guide / Privacy    Help, scholarship, local  controls     ASSURANCE    Focused tests    Generation, runtime, save,  a11y, viewport     ASSURANCE    Evidence records    Simulation runs, status, route  checks     PUBLICATION    Notebook publication    Executed sources, HTML,  manifests         Data/state dependency      Runtime control      Verification/publication dependency    Validated: 12 nodes · 9 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Scenario grammar : Incidents, routes, artifacts, choices   Actor generation : Motives, affect, ties, repertoires   Coherence gates : Schema, agency, ethics, replay readiness   Night engine : Clock, access, reducer, completion   Cross-room effects : Directed ambient and direct receipts   State and persistence : Night state, save schema, digest   UI shell : One viewport and room navigation   Trace / Relations / Receipts : Bounded inspection and interpretation   House Guide / Privacy : Help, scholarship, local controls   Focused tests : Generation, runtime, save, a11y, viewport   Evidence records : Simulation runs, status, route checks   Notebook publication : Executed sources, HTML, manifests   Relationships   Scenario grammar → Night engine  Actor generation → Cross-room effects  Coherence gates → State and persistence  Night engine → UI shell  Cross-room effects → Trace / Relations / Receipts  State and persistence → House Guide / Privacy  UI shell → Focused tests  Trace / Relations / Receipts → Evidence records  House Guide / Privacy → Notebook publication

### Build and publication pipeline

One deterministic standard-library builder executes notebook cells, renders the HTML editions, verifies identities and hashes, and publishes the same records into public routes and the production distribution.

In [6]:
nodes=[
 dnode('source',30,210,190,100,'Notebook specifications','Four declarative cell sets in the standard-library builder','component','document','source'),
 dnode('builder',280,210,200,100,'Deterministic builder','Fresh namespace, executed cells, stable IDs','component','rect','build step'),
 dnode('ipynb',560,80,210,100,'Executed .ipynb','Canonical source with committed outputs','data','document','artifact'),
 dnode('html',560,330,210,100,'Styled HTML edition','Self-contained, script-free, accessible route','evidence','document','artifact'),
 dnode('manifest',850,205,210,110,'Manifest and drift gate','Bytes, SHA-256, source-copy identity','evidence','document','verification'),
 dnode('public',1130,120,220,100,'public/notebooks/','Downloads, HTML index, direct routes','system','rect','publication'),
 dnode('dist',1130,330,220,100,'Production dist/','Built client assets and Worker routes','system','rect','deployment artifact'),
]
edges=[
 dedge('source','builder',[(220,260),(280,260)],'data'),
 dedge('builder','ipynb',[(480,240),(520,240),(520,130),(560,130)],'data','execute'),
 dedge('builder','html',[(480,280),(530,280),(530,380),(560,380)],'evidence','render'),
 dedge('ipynb','manifest',[(770,130),(810,130),(810,245),(850,245)],'evidence'),
 dedge('html','manifest',[(770,380),(820,380),(820,275),(850,275)],'evidence'),
 dedge('manifest','public',[(1060,240),(1090,240),(1090,170),(1130,170)],'evidence','publish'),
 dedge('manifest','dist',[(1060,270),(1100,270),(1100,380),(1130,380)],'evidence','build'),
]
_html = diagram_html('Build and publication pipeline','Deterministic notebook build and publication pipeline','One builder produces executed notebook source and styled HTML, verifies their identity and hashes, then publishes the same records into public routes and the production distribution.',1390,520,nodes,edges,legend=[('data','Executed source'),('evidence','Rendered or verified artifact')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Build and publication pipeline  Deterministic notebook build and publication pipeline  One builder produces executed notebook source and styled HTML, verifies their identity and hashes, then publishes the same records into public routes and the production distribution.     Deterministic notebook build and publication pipeline  One builder produces executed notebook source and styled HTML, verifies their identity and hashes, then publishes the same records into public routes and the production distribution.               execute      render        publish      build     SOURCE    Notebook specifications    Four declarative cell  sets in the  standard-library  builder     BUILD STEP    Deterministic builder    Fresh namespace,  executed cells, stable  IDs     ARTIFACT    Executed .ipynb    Canonical source with  committed outputs     ARTIFACT    Styled HTML edition    Self-contained,  script-free, accessible  route     VERIFICATION    Manifest and drift gate    Bytes, SHA-256,  source-copy identity     PUBLICATION    public/notebooks/    Downloads, HTML index,  direct routes     DEPLOYMENT ARTIFACT    Production dist/    Built client assets and  Worker routes         Executed source      Rendered or verified artifact    Validated: 7 nodes · 7 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    Notebook specifications : Four declarative cell sets in the standard-library builder   Deterministic builder : Fresh namespace, executed cells, stable IDs   Executed .ipynb : Canonical source with committed outputs   Styled HTML edition : Self-contained, script-free, accessible route   Manifest and drift gate : Bytes, SHA-256, source-copy identity   public/notebooks/ : Downloads, HTML index, direct routes   Production dist/ : Built client assets and Worker routes   Relationships   Notebook specifications → Deterministic builder  Deterministic builder → Executed .ipynb (execute)  Deterministic builder → Styled HTML edition (render)  Executed .ipynb → Manifest and drift gate  Styled HTML edition → Manifest and drift gate  Manifest and drift gate → public/notebooks/ (publish)  Manifest and drift gate → Production dist/ (build)

### Artifact publication structure

Canonical notebooks, public source copies, static HTML editions, and integrity metadata are separate artifact families that move together. The structure makes ownership, reproducibility, and drift detection visible.

In [7]:
nodes=[
 dnode('builder-root',300,40,800,100,'scripts/docs/build_notebooks.py','Single canonical specification and execution authority','component','rect','publication root'),
 dnode('canonical',80,230,220,100,'notebooks/','Four canonical executed .ipynb sources','data','document','source collection'),
 dnode('downloads',380,230,220,100,'public/notebooks/*.ipynb','Byte-identical downloadable copies','data','document','published source'),
 dnode('htmls',680,230,220,100,'public/notebooks/*.html','Four self-contained styled editions','evidence','document','published edition'),
 dnode('manifest',980,230,220,100,'artifact-manifest.json','Path, media type, bytes, SHA-256','evidence','document','integrity'),
 dnode('canon-detail',80,470,220,90,'Model · Research · Systems · Validation','Stable notebook filenames and cell outputs','component','rect','contents'),
 dnode('download-detail',380,470,220,90,'Direct source downloads','Inspect, rerun, and compare committed outputs','system','rect','route behavior'),
 dnode('html-detail',680,470,220,90,'Notebook index and direct pages','Long-form semantic reading outside the game shell','system','rect','route behavior'),
 dnode('manifest-detail',980,470,220,90,'Drift and identity checks','Build fails when committed artifacts differ','evidence','rect','assurance'),
]
edges=[
 dedge('builder-root','canonical',[(370,140),(370,155),(190,155),(190,230)],'data'),
 dedge('builder-root','downloads',[(570,140),(570,170),(490,170),(490,230)],'data'),
 dedge('builder-root','htmls',[(770,140),(770,185),(790,185),(790,230)],'evidence'),
 dedge('builder-root','manifest',[(970,140),(970,200),(1090,200),(1090,230)],'evidence'),
 dedge('canonical','canon-detail',[(190,330),(190,470)],'association'),
 dedge('downloads','download-detail',[(490,330),(490,470)],'association'),
 dedge('htmls','html-detail',[(790,330),(790,470)],'association'),
 dedge('manifest','manifest-detail',[(1090,330),(1090,470)],'association'),
]
_html = diagram_html('Artifact publication structure diagram','Notebook artifact publication structure','The builder owns four coordinated artifact families. Canonical sources, download copies, static editions, and integrity metadata move together and have distinct responsibilities.',1280,620,nodes,edges,legend=[('data','Notebook source'),('evidence','Published or integrity artifact'),('association','Contains / elaborates')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Artifact publication structure diagram  Notebook artifact publication structure  The builder owns four coordinated artifact families. Canonical sources, download copies, static editions, and integrity metadata move together and have distinct responsibilities.     Notebook artifact publication structure  The builder owns four coordinated artifact families. Canonical sources, download copies, static editions, and integrity metadata move together and have distinct responsibilities.                        PUBLICATION ROOT    scripts/docs/build_notebooks.py    Single canonical specification and execution authority     SOURCE COLLECTION    notebooks/    Four canonical executed  .ipynb sources     PUBLISHED SOURCE    public/notebooks/*.ipynb    Byte-identical downloadable  copies     PUBLISHED EDITION    public/notebooks/*.html    Four self-contained styled  editions     INTEGRITY    artifact-manifest.json    Path, media type, bytes,  SHA-256     CONTENTS    Model · Research · Systems  · Validation    Stable notebook filenames  and cell outputs     ROUTE BEHAVIOR    Direct source downloads    Inspect, rerun, and compare  committed outputs     ROUTE BEHAVIOR    Notebook index and direct  pages    Long-form semantic reading  outside the game shell     ASSURANCE    Drift and identity checks    Build fails when committed  artifacts differ         Notebook source      Published or integrity artifact      Contains / elaborates    Validated: 9 nodes · 8 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    scripts/docs/build_notebooks.py : Single canonical specification and execution authority   notebooks/ : Four canonical executed .ipynb sources   public/notebooks/*.ipynb : Byte-identical downloadable copies   public/notebooks/*.html : Four self-contained styled editions   artifact-manifest.json : Path, media type, bytes, SHA-256   Model · Research · Systems · Validation : Stable notebook filenames and cell outputs   Direct source downloads : Inspect, rerun, and compare committed outputs   Notebook index and direct pages : Long-form semantic reading outside the game shell   Drift and identity checks : Build fails when committed artifacts differ   Relationships   scripts/docs/build_notebooks.py → notebooks/  scripts/docs/build_notebooks.py → public/notebooks/*.ipynb  scripts/docs/build_notebooks.py → public/notebooks/*.html  scripts/docs/build_notebooks.py → artifact-manifest.json  notebooks/ → Model · Research · Systems · Validation  public/notebooks/*.ipynb → Direct source downloads  public/notebooks/*.html → Notebook index and direct pages  artifact-manifest.json → Drift and identity checks

### Route and document relationships

The route map distinguishes the fixed application, the House Guide reading alert, standalone notebook pages, evidence pages, and the downloadable notebook sources paired with each rendered edition.

In [8]:
nodes=[
 dnode('app',30,300,180,100,'/','CHORUS application route','system','rect','entry'),
 dnode('guide',280,100,230,100,'House Guide / Field Notes','In-app alert and named iframe','system','rect','in-app surface'),
 dnode('index',280,300,230,100,'/notebooks/index.html','Standalone record index','system','rect','public route'),
 dnode('evidence-index',280,500,230,100,'/evidence/index.html','Assurance and status index','system','rect','public route'),
 dnode('inapp',650,100,270,100,'In-app scholarly viewer','Model Specification and Research Design','evidence','rect','embedded document'),
 dnode('standalone',650,280,270,140,'Four standalone records','Systems · Model · Research · Validation','evidence','rect','document routes'),
 dnode('status',650,500,270,100,'Current and historical evidence','Working-tree status, retained runs, route checks','evidence','rect','evidence routes'),
 dnode('downloads-r',1050,280,250,140,'Downloadable .ipynb sources','Same four records beside HTML editions','data','document','source routes'),
]
edges=[
 dedge('app','guide',[(210,325),(240,325),(240,150),(280,150)],'flow','opens'),
 dedge('app','index',[(210,350),(280,350)],'flow','links'),
 dedge('app','evidence-index',[(210,375),(250,375),(250,550),(280,550)],'flow','links'),
 dedge('guide','inapp',[(510,150),(650,150)],'evidence','loads'),
 dedge('index','standalone',[(510,350),(650,350)],'evidence','lists'),
 dedge('evidence-index','status',[(510,550),(650,550)],'evidence','lists'),
 dedge('standalone','downloads-r',[(920,350),(1050,350)],'data','pairs with'),
]
_html = diagram_html('Route and document relationship map','Application, notebook, and evidence routes','The fixed game route opens scholarship inside House Guide while the notebook and evidence indexes expose long-form standalone documents. Every HTML edition remains paired with its executed notebook source.',1340,670,nodes,edges,legend=[('flow','Application navigation'),('evidence','Published document relation'),('data','Source-download relation')])
print("PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.")
_html

PASS: deterministic SVG diagram rendered; orthogonal routing and node separation validated.


Route and document relationship map  Application, notebook, and evidence routes  The fixed game route opens scholarship inside House Guide while the notebook and evidence indexes expose long-form standalone documents. Every HTML edition remains paired with its executed notebook source.     Application, notebook, and evidence routes  The fixed game route opens scholarship inside House Guide while the notebook and evidence indexes expose long-form standalone documents. Every HTML edition remains paired with its executed notebook source.                 opens      links      links      loads      lists      lists      pairs with     ENTRY    /    CHORUS application  route     IN-APP SURFACE    House Guide / Field Notes    In-app alert and named iframe     PUBLIC ROUTE    /notebooks/index.html    Standalone record index     PUBLIC ROUTE    /evidence/index.html    Assurance and status index     EMBEDDED DOCUMENT    In-app scholarly viewer    Model Specification and Research  Design     DOCUMENT ROUTES    Four standalone records    Systems · Model · Research ·  Validation     EVIDENCE ROUTES    Current and historical evidence    Working-tree status, retained  runs, route checks     SOURCE ROUTES    Downloadable .ipynb sources    Same four records beside HTML  editions         Application navigation      Published document relation      Source-download relation    Validated: 8 nodes · 7 edges · orthogonal routing · 0 crossings · 0 node incursions · 0 node overlaps   Text equivalent  Elements    / : CHORUS application route   House Guide / Field Notes : In-app alert and named iframe   /notebooks/index.html : Standalone record index   /evidence/index.html : Assurance and status index   In-app scholarly viewer : Model Specification and Research Design   Four standalone records : Systems · Model · Research · Validation   Current and historical evidence : Working-tree status, retained runs, route checks   Downloadable .ipynb sources : Same four records beside HTML editions   Relationships   / → House Guide / Field Notes (opens)  / → /notebooks/index.html (links)  / → /evidence/index.html (links)  House Guide / Field Notes → In-app scholarly viewer (loads)  /notebooks/index.html → Four standalone records (lists)  /evidence/index.html → Current and historical evidence (lists)  Four standalone records → Downloadable .ipynb sources (pairs with)

## Concurrent-night geometry

Room switching is navigation, not turn-taking. A decision advances the shared clock and writes one local effect plus one bounded effect for each other room. Direct content transfer is rarer than systemic influence and must pass its own compatibility gate.

In [9]:
geometry = {
    "rooms": 6,
    "beats_per_room": 4,
    "decisions": 6 * 4,
    "effects_per_decision": 6,
    "decision_effect_receipts": 6 * 4 * 6,
    "ordered_room_pairs": 6 * 5,
    "sparse_direct_crossings": 6,
}
cards = [
    ("Rooms", geometry["rooms"], "All exist from minute zero."),
    ("Decisions", geometry["decisions"], "Four causal beats in each room."),
    ("Decision receipts", geometry["decision_effect_receipts"], "One local and five remote per decision."),
    ("Directed routes", geometry["ordered_room_pairs"], "Every ordered room pair has a bounded systemic route."),
    ("Direct crossings", geometry["sparse_direct_crossings"], "Only compatibility-checked content crossings."),
]
_html = cards_html("Concurrent-night fixed geometry", cards)
assert geometry["decision_effect_receipts"] == 144
assert geometry["ordered_room_pairs"] == 30
print("PASS: 24 decisions yield 144 decision-effect receipts across 30 directed room pairs.")
_html

PASS: 24 decisions yield 144 decision-effect receipts across 30 directed room pairs.


Rooms  6  All exist from minute zero.    Decisions  24  Four causal beats in each room.    Decision receipts  144  One local and five remote per decision.    Directed routes  30  Every ordered room pair has a bounded systemic route.    Direct crossings  6  Only compatibility-checked content crossings.

In [10]:
propagation = [
    ("Local effect", "Always", "The occupied room's metrics and ledgers", "Accepted decision"),
    ("Ambient systemic effect", "Five per decision", "Pressure, reach, fatigue, trust, or repair conditions", "Typed directed route"),
    ("Direct crossing", "Sparse", "A supported fragment or recognizable carrier", "Shared channel or artifact compatibility"),
    ("Scheduled pulse", "Once when due", "Background change independent of visit order", "Shared logical clock"),
    ("Afterimage", "After room close", "Later incoming effects without rewriting close metrics", "Immutable completion snapshot"),
]
_html = table_html("Propagation layers and their evidence requirements", ("Layer", "Frequency", "May change", "Required basis"), propagation, row_headers=True)
print("Propagation layers remain distinct; ambient influence never asserts shared content.")
_html

Propagation layers remain distinct; ambient influence never asserts shared content.


Layer,Frequency,May change,Required basis
Local effect,Always,The occupied room's metrics and ledgers,Accepted decision
Ambient systemic effect,Five per decision,"Pressure, reach, fatigue, trust, or repair conditions",Typed directed route
Direct crossing,Sparse,A supported fragment or recognizable carrier,Shared channel or artifact compatibility
Scheduled pulse,Once when due,Background change independent of visit order,Shared logical clock
Afterimage,After room close,Later incoming effects without rewriting close metrics,Immutable completion snapshot


## Evidence and interpretation boundaries

The simulation can represent motive because the player occupies a fictional seat. That authored interior is still separate from the incident record. Presentation, relationship, class position, register, group status, and audience reaction are never promoted into facts about conduct on their own.

In [11]:
layers = [
    ("Ground truth", "Fixed incident facts", "Generation only", "Never altered by play"),
    ("Observation", "Represented conduct or artifact", "Source-backed scene record", "May support bounded correction"),
    ("Audience inference", "What others make the conduct mean", "Relational ledger", "Cannot become fact without evidence"),
    ("Authored interior", "Occupied protagonist motive and emotional load", "Seat model", "Explains access; does not excuse choice"),
    ("Unknown", "Unresolved cause, intent, or scope", "Explicit uncertainty ledger", "Must remain unresolved until supported"),
]
_html = table_html("Evidence and interpretation layers", ("Layer", "Contains", "Authority", "Prohibition"), layers, row_headers=True)
print("PASS: five layers retain distinct authority and correction rules.")
_html

PASS: five layers retain distinct authority and correction rules.


Layer,Contains,Authority,Prohibition
Ground truth,Fixed incident facts,Generation only,Never altered by play
Observation,Represented conduct or artifact,Source-backed scene record,May support bounded correction
Audience inference,What others make the conduct mean,Relational ledger,Cannot become fact without evidence
Authored interior,Occupied protagonist motive and emotional load,Seat model,Explains access; does not excuse choice
Unknown,"Unresolved cause, intent, or scope",Explicit uncertainty ledger,Must remain unresolved until supported


In [12]:
communication = [
    ("Linguistic repertoire", "Several learned registers available to one actor", "Region, family, peers, profession, institution, politics, platform", "Does not determine belief or worth"),
    ("Register action", "Maintain, bridge, or switch for a represented audience", "Scene, relationship, pressure, channel", "Does not prove deceit"),
    ("Mental model", "Assumptions about care, evidence, authority, disagreement, responsibility", "Actor-level world model", "May diverge even under a shared register"),
    ("Presentation temperature", "Warm or cool surface in one exchange", "Bounded scene wording", "Is not a stable personality type"),
]
_html = table_html("Communication model separations", ("Construct", "Representation", "Inputs", "Boundary"), communication, row_headers=True)
print("Communication codes, world models, motives, and truth remain independently represented.")
_html

Communication codes, world models, motives, and truth remain independently represented.


Construct,Representation,Inputs,Boundary
Linguistic repertoire,Several learned registers available to one actor,"Region, family, peers, profession, institution, politics, platform",Does not determine belief or worth
Register action,"Maintain, bridge, or switch for a represented audience","Scene, relationship, pressure, channel",Does not prove deceit
Mental model,"Assumptions about care, evidence, authority, disagreement, responsibility",Actor-level world model,May diverge even under a shared register
Presentation temperature,Warm or cool surface in one exchange,Bounded scene wording,Is not a stable personality type


## Choice access and fatigue

Discernment and enactment are separate. A seat can continue to recognize the sound action while accumulated platform load makes that action temporarily impossible to carry. An attempted blocked ideal produces a concise explanation inside that choice pane and does not mutate the night.

In [13]:
access = [
    ("Visible", "Beat has arrived", "Future choice content remains sealed before arrival."),
    ("Structurally available", "Required source, authority, evidence, relationship, and distribution supports exist", "Prior decisions in other rooms may assemble support."),
    ("Enactable", "Required follow-through is within remaining modeled capacity", "Fatigue can block enactment without lowering discernment."),
    ("Accepted", "Scene is current, event is unique, access passes", "Reducer writes one decision and six effect receipts."),
    ("Non-amplification floor", "Always enactable", "The player is never forced to repeat, personalize, or spread a claim."),
]
_html = table_html("Choice access gates", ("Gate", "Condition", "System promise"), access, row_headers=True)
print("PASS: access separates arrival, structural support, follow-through capacity, and acceptance.")
_html

PASS: access separates arrival, structural support, follow-through capacity, and acceptance.


Gate,Condition,System promise
Visible,Beat has arrived,Future choice content remains sealed before arrival.
Structurally available,"Required source, authority, evidence, relationship, and distribution supports exist",Prior decisions in other rooms may assemble support.
Enactable,Required follow-through is within remaining modeled capacity,Fatigue can block enactment without lowering discernment.
Accepted,"Scene is current, event is unique, access passes",Reducer writes one decision and six effect receipts.
Non-amplification floor,Always enactable,"The player is never forced to repeat, personalize, or spread a claim."


In [14]:
fatigue = [
    ("Attentional", "Competing artifacts and context switches", "Focus cost"),
    ("Affective", "Repeated urgency, outrage, and anticipated threat", "Alarm cost"),
    ("Relational", "Continuous calculation of tone, loyalty, and reply cost", "Social cost"),
    ("Verification", "Source recovery across fragmented copies", "Checking cost"),
    ("Efficacy", "Repeated experience of repair lagging behind spread", "Action-will cost"),
]
_html = table_html("Modeled fatigue channels", ("Channel", "Accumulation source", "Primary load"), fatigue, row_headers=True)
print("Five fatigue channels affect enactment; none reduces the seat's discernment metric.")
_html

Five fatigue channels affect enactment; none reduces the seat's discernment metric.


Channel,Accumulation source,Primary load
Attentional,Competing artifacts and context switches,Focus cost
Affective,"Repeated urgency, outrage, and anticipated threat",Alarm cost
Relational,"Continuous calculation of tone, loyalty, and reply cost",Social cost
Verification,Source recovery across fragmented copies,Checking cost
Efficacy,Repeated experience of repair lagging behind spread,Action-will cost


## Persistence, privacy, and replay

The baseline session remains ephemeral. Local slots require an explicit choice, and portable export is a player-directed file operation. A canonical replay restores the entire night because a single-room rewind would break concurrency and causal provenance.

In [15]:
persistence = [
    ("Session only", "Default", "Memory for the current browser session", "Nothing written to a slot"),
    ("Local slot", "Explicit per-slot consent", "Versioned complete night state", "Inspect and delete controls"),
    ("Portable export", "Explicit download", "Text envelope, schema, provenance, digest", "512 KiB maximum on import"),
    ("Import", "Player-selected file", "Parse, migrate, validate, preview, restore", "No mutation before validation"),
    ("Replay", "Whole-night state", "Seed plus complete ordered event record", "No isolated room rewind"),
]
_html = table_html("Persistence and replay contract", ("Mode", "Consent", "Contents", "Boundary"), persistence, row_headers=True)
print("Persistence remains local, bounded, versioned, and reversible by the player.")
_html

Persistence remains local, bounded, versioned, and reversible by the player.


Mode,Consent,Contents,Boundary
Session only,Default,Memory for the current browser session,Nothing written to a slot
Local slot,Explicit per-slot consent,Versioned complete night state,Inspect and delete controls
Portable export,Explicit download,"Text envelope, schema, provenance, digest",512 KiB maximum on import
Import,Player-selected file,"Parse, migrate, validate, preview, restore",No mutation before validation
Replay,Whole-night state,Seed plus complete ordered event record,No isolated room rewind


## Maintenance seams

Changes should begin at the narrowest authoritative layer. New narrative grammar belongs in the generator; new transition behavior belongs in the reducer; new disclosure belongs in the renderer only after the underlying state already exists.

In [16]:
maintenance = [
    ("Add a room grammar", "Generator", "Coherence report, six-dynamic coverage, youth safety, route compatibility"),
    ("Add a metric", "Generator and reducer", "Bounds, local effect, remote effect, persistence, receipt wording"),
    ("Add a fatigue channel", "Types, access checks, receipts", "Discernment separation, caps, blocked reason, save validation"),
    ("Add a cross-room mechanism", "Link grammar and reducer", "All 30 pairs, direct/ambient boundary, reveal policy"),
    ("Add a disclosure", "Renderer", "Arrival gate, progressive disclosure, keyboard path, mobile fit"),
]
_html = table_html("Change routing and required companion work", ("Change", "Authoritative layer", "Release companions"), maintenance, row_headers=True)
print("Maintenance map complete: each change names its authority and release companions.")
_html

Maintenance map complete: each change names its authority and release companions.


Change,Authoritative layer,Release companions
Add a room grammar,Generator,"Coherence report, six-dynamic coverage, youth safety, route compatibility"
Add a metric,Generator and reducer,"Bounds, local effect, remote effect, persistence, receipt wording"
Add a fatigue channel,"Types, access checks, receipts","Discernment separation, caps, blocked reason, save validation"
Add a cross-room mechanism,Link grammar and reducer,"All 30 pairs, direct/ambient boundary, reveal policy"
Add a disclosure,Renderer,"Arrival gate, progressive disclosure, keyboard path, mobile fit"
